<a href="https://colab.research.google.com/github/juergenlandauer/FoundationModelsArchaeology/blob/main/image_downloading_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Image Downloader



In [1]:
# some libraries we use
!pip install -Uq rasterio

## Imports

Import the required libraries for image processing, HTTP requests, and multithreading.

In [2]:
# Input CSV file path (should have 'lat' and 'lon' columns)
CSV_FILE = 'random_coordinates.csv'

# Image parameters
TILE_SIZE_PIXELS = 512  # Width/Height in PIXELS (e.g., 512 x 512 pixels)
ZOOM_LEVEL = 18         # Zoom level (1-20, higher = more detailed)
MAP_TYPE = 'satellite'  # 'satellite', 'roadmap', 'hybrid', 'terrain'

# Output parameters
OUTPUT_DIR = 'satellite_patches'
OUTPUT_FORMAT = 'tif'  # use 'tif' or 'png'

# Delay between requests (seconds) to avoid rate limiting
REQUEST_DELAY = 0.5

## Configuration Parameters

**Important**: `TILE_SIZE_PIXELS` now represents the desired image size in pixels (not meters on the ground).

- **TILE_SIZE_PIXELS**: The desired output image size in pixels (e.g., 512 means 512px × 512px)
- **ZOOM_LEVEL**: Controls the level of detail and ground coverage. Higher zoom = more detail, less ground area
  - At zoom 19, approximately 0.29 meters/pixel at 24° latitude
  - At zoom 20, approximately 0.14 meters/pixel at 24° latitude

The actual ground coverage in meters will vary based on:
1. The `TILE_SIZE_PIXELS` setting (image size)
2. The `ZOOM_LEVEL` (affects ground resolution)
3. The latitude (affects ground resolution due to Web Mercator projection)

In [3]:
import pandas as pd
# Load the CSV file
df = pd.read_csv(CSV_FILE)

# Try to identify lat/lon columns
lat_col = None
lon_col = None

for col in df.columns:
    col_lower = col.lower()
    if 'lat' in col_lower and lat_col is None:
        lat_col = col
    if 'lon' in col_lower and lon_col is None:
        lon_col = col

if lat_col is None or lon_col is None:
    raise ValueError(f"Could not find latitude and longitude columns. Available columns: {df.columns.tolist()}")

print(f"Found {len(df)} coordinates")
print(f"Using columns: {lat_col}, {lon_col}")
display(df.head())

Found 500 coordinates
Using columns: lat, lon


,lat,lon
0,23.944919,85.543857
1,24.288923,88.455822
2,23.506138,86.745067
3,24.888847,88.247951
4,25.658659,85.697537


In [ ]:
# in case we need to swap columns
df = df.rename(columns={ 'lat':'lon', 'lon': 'lat'})
df.head()

,lat,lon
0,24.257411,88.247931
1,24.247894,88.240021
2,22.720882,88.369454
3,23.171771,87.229147
4,23.897020,87.537847


In [4]:
import cv2
import requests
import numpy as np
import threading

## Helper Functions

### Download Tile

Downloads a single tile from the given URL and decodes it into a numpy array.

In [5]:
def download_tile(url, headers, channels):
    response = requests.get(url, headers=headers)
    arr =  np.asarray(bytearray(response.content), dtype=np.uint8)

    if channels == 3:
        return cv2.imdecode(arr, 1)
    return cv2.imdecode(arr, -1)

### Mercator Projection

Projects latitude/longitude coordinates to pixel coordinates using the Mercator projection.

Reference: https://developers.google.com/maps/documentation/javascript/examples/map-coordinates

In [6]:
def project_with_scale(lat, lon, scale):
    siny = np.sin(lat * np.pi / 180)
    siny = min(max(siny, -0.9999), 0.9999)
    x = scale * (0.5 + lon / 360)
    y = scale * (0.5 - np.log((1 + siny) / (1 - siny)) / (4 * np.pi))
    return x, y

## Main Download Function

Downloads a map region and stitches tiles together into a single image.

### Parameters:
- `(lat1, lon1)` - Coordinates (decimal degrees) of the top-left corner
- `(lat2, lon2)` - Coordinates (decimal degrees) of the bottom-right corner
- `zoom` - Zoom level
- `url` - Tile URL with {x}, {y} and {z} placeholders
- `headers` - Dictionary of HTTP headers
- `tile_size` - Server tile size in pixels (default: 256 for Google Maps)
- `channels` - Number of channels in the output image (default: 3 for BGR)

### Returns:
A `numpy.ndarray` containing the downloaded image in BGR or BGRA format.

In [7]:
def download_image(lat1: float, lon1: float, lat2: float, lon2: float,
    zoom: int, url: str, headers: dict, tile_size: int = 256, channels: int = 3) -> np.ndarray:
    """
    Downloads a map region. Returns an image stored as a `numpy.ndarray` in BGR or BGRA, depending on the number
    of `channels`.

    Parameters
    ----------
    `(lat1, lon1)` - Coordinates (decimal degrees) of the top-left corner of a rectangular area

    `(lat2, lon2)` - Coordinates (decimal degrees) of the bottom-right corner of a rectangular area

    `zoom` - Zoom level

    `url` - Tile URL with {x}, {y} and {z} in place of its coordinate and zoom values

    `headers` - Dictionary of HTTP headers

    `tile_size` - Tile size in pixels

    `channels` - Number of channels in the output image. Also affects how the tiles are converted into numpy arrays.
    """

    scale = 1 << zoom

    # Find the pixel coordinates and tile coordinates of the corners
    tl_proj_x, tl_proj_y = project_with_scale(lat1, lon1, scale)
    br_proj_x, br_proj_y = project_with_scale(lat2, lon2, scale)

    tl_pixel_x = int(tl_proj_x * tile_size)
    tl_pixel_y = int(tl_proj_y * tile_size)
    br_pixel_x = int(br_proj_x * tile_size)
    br_pixel_y = int(br_proj_y * tile_size)

    tl_tile_x = int(tl_proj_x)
    tl_tile_y = int(tl_proj_y)
    br_tile_x = int(br_proj_x)
    br_tile_y = int(br_proj_y)

    img_w = abs(tl_pixel_x - br_pixel_x)
    img_h = br_pixel_y - tl_pixel_y
    img = np.zeros((img_h, img_w, channels), np.uint8)


    def build_row(tile_y):
        for tile_x in range(tl_tile_x, br_tile_x + 1):
            tile = download_tile(url.format(x=tile_x, y=tile_y, z=zoom), headers, channels)

            if tile is not None:
                # Find the pixel coordinates of the new tile relative to the image
                tl_rel_x = tile_x * tile_size - tl_pixel_x
                tl_rel_y = tile_y * tile_size - tl_pixel_y
                br_rel_x = tl_rel_x + tile_size
                br_rel_y = tl_rel_y + tile_size

                # Define where the tile will be placed on the image
                img_x_l = max(0, tl_rel_x)
                img_x_r = min(img_w + 1, br_rel_x)
                img_y_l = max(0, tl_rel_y)
                img_y_r = min(img_h + 1, br_rel_y)

                # Define how border tiles will be cropped
                cr_x_l = max(0, -tl_rel_x)
                cr_x_r = tile_size + min(0, img_w - br_rel_x)
                cr_y_l = max(0, -tl_rel_y)
                cr_y_r = tile_size + min(0, img_h - br_rel_y)

                img[img_y_l:img_y_r, img_x_l:img_x_r] = tile[cr_y_l:cr_y_r, cr_x_l:cr_x_r]


    threads = []
    for tile_y in range(tl_tile_y, br_tile_y + 1):
        thread = threading.Thread(target=build_row, args=[tile_y])
        thread.start()
        threads.append(thread)

    for thread in threads:
        thread.join()

    return img

## Image Size Calculator

Calculates the size of an image without downloading it. Returns the width and height in pixels as a tuple.

In [8]:
def image_size(lat1: float, lon1: float, lat2: float,
    lon2: float, zoom: int, tile_size: int = 256):
    """ Calculates the size of an image without downloading it. Returns the width and height in pixels as a tuple. """

    scale = 1 << zoom
    tl_proj_x, tl_proj_y = project_with_scale(lat1, lon1, scale)
    br_proj_x, br_proj_y = project_with_scale(lat2, lon2, scale)

    tl_pixel_x = int(tl_proj_x * tile_size)
    tl_pixel_y = int(tl_proj_y * tile_size)
    br_pixel_x = int(br_proj_x * tile_size)
    br_pixel_y = int(br_proj_y * tile_size)

    return abs(tl_pixel_x - br_pixel_x), br_pixel_y - tl_pixel_y

## Default Preferences

Configuration settings for tile server and headers.

In [9]:
default_prefs = {
    'url': 'https://mt.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
    'tile_size_pixels': 256,  # Tile size from the server in pixels (always 256 for Google Maps)
    'channels': 3,
    'headers': {
        'cache-control': 'max-age=0',
        'sec-ch-ua': '" Not A;Brand";v="99", "Chromium";v="99", "Google Chrome";v="99"',
        'sec-ch-ua-mobile': '?0',
        'sec-ch-ua-platform': '"Windows"',
        'sec-fetch-dest': 'document',
        'sec-fetch-mode': 'navigate',
        'sec-fetch-site': 'none',
        'sec-fetch-user': '?1',
        'upgrade-insecure-requests': '1',
        'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/99.0.4844.82 Safari/537.36'
    }
}

## Ground Resolution Calculator

This function calculates the ground resolution (meters per pixel) at a given latitude and zoom level.

**Formula**: At the equator, ground resolution = (Earth circumference) / (256 × 2^zoom)
At other latitudes, multiply by cos(latitude) to account for meridian convergence.

In [10]:
def ground_resolution(lat, zoom):
    """
    Calculate the ground resolution (meters per pixel) at a given latitude and zoom level.

    At the equator, the circumference of Earth is ~40,075,017 meters.
    At zoom level z, the world is represented by 256 * 2^z pixels.

    Parameters:
    - lat: latitude in decimal degrees
    - zoom: zoom level (1-20)

    Returns:
    - Ground resolution in meters per pixel
    """
    earth_circumference = 40075017  # meters at equator
    pixels_at_zoom = 256 * (2 ** zoom)
    meters_per_pixel_equator = earth_circumference / pixels_at_zoom

    # Adjust for latitude (meridians converge at poles)
    meters_per_pixel = meters_per_pixel_equator * np.cos(np.radians(lat))

    return meters_per_pixel

# Display ground resolution for current settings
sample_lat = 24.0  # approximate latitude for India
resolution = ground_resolution(sample_lat, ZOOM_LEVEL)
print(f"Ground Resolution at zoom {ZOOM_LEVEL} and latitude {sample_lat}°:")
print(f"  {resolution:.4f} meters/pixel")
print(f"  {1/resolution:.2f} pixels/meter")
print(f"\nFor a {TILE_SIZE_PIXELS}px x {TILE_SIZE_PIXELS}px image:")
expected_meters = TILE_SIZE_PIXELS * resolution
print(f"  Ground coverage: ~{expected_meters:.1f}m x {expected_meters:.1f}m")

Ground Resolution at zoom 18 and latitude 24.0°:
  0.5455 meters/pixel
  1.83 pixels/meter

For a 512px x 512px image:
  Ground coverage: ~279.3m x 279.3m


In [ ]:
import os
# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Earth's radius in meters
EARTH_RADIUS = 6371000

def pixels_to_lat_lon_offset(pixels, lat, zoom):
    """
    Convert a distance in pixels to lat/lon degree offsets, considering zoom level.

    Parameters:
    - pixels: distance in pixels
    - lat: latitude at which to compute the offset (affects ground resolution)
    - zoom: zoom level (affects ground resolution)

    Returns:
    - lat_offset: offset in degrees for latitude
    - lon_offset: offset in degrees for longitude
    """
    # Calculate ground resolution (meters per pixel) at this latitude and zoom
    resolution = ground_resolution(lat, zoom)

    # Convert pixels to meters
    meters = pixels * resolution

    # Convert meters to degree offsets
    # 1 degree of latitude is approximately 111,320 meters everywhere
    lat_offset = meters / 111320.0

    # 1 degree of longitude varies with latitude
    # At the equator it's ~111,320m, at poles it's 0
    lon_offset = meters / (111320.0 * np.cos(np.radians(lat)))

    return lat_offset, lon_offset

# Iterate over each row in the dataframe
for idx, row in df.iterrows():
    lat = row[lat_col]
    lon = row[lon_col]

    # Calculate the offset for the desired tile size in pixels
    # TILE_SIZE_PIXELS defines the edge length of the square area in pixels
    half_edge_pixels = TILE_SIZE_PIXELS / 2
    lat_offset, lon_offset = pixels_to_lat_lon_offset(half_edge_pixels, lat, ZOOM_LEVEL)

    # Compute bounding box with (lat, lon) as center
    lat1 = lat + lat_offset  # North (top)
    lon1 = lon - lon_offset  # West (left)
    lat2 = lat - lat_offset  # South (bottom)
    lon2 = lon + lon_offset  # East (right)

    # Calculate ground coverage
    resolution = ground_resolution(lat, ZOOM_LEVEL)
    ground_coverage = TILE_SIZE_PIXELS * resolution

    print(f"Processing {idx + 1}/{len(df)}: Center ({lat:.6f}, {lon:.6f})")
    print(f"  Bounding box: ({lat1:.6f}, {lon1:.6f}) to ({lat2:.6f}, {lon2:.6f})")
    print(f"  Target image size: {TILE_SIZE_PIXELS}px x {TILE_SIZE_PIXELS}px at zoom level {ZOOM_LEVEL}")
    print(f"  Ground coverage: ~{ground_coverage:.1f}m x {ground_coverage:.1f}m (at {resolution:.4f} m/pixel)")

    # Calculate expected image size in pixels
    # The tile_size parameter here is the server's tile pixel size (256)
    width, height = image_size(lat1, lon1, lat2, lon2, ZOOM_LEVEL, default_prefs['tile_size_pixels'])
    print(f"  Expected image size: {width}x{height} pixels")

    # Download the image
    img = download_image(
        lat1, lon1, lat2, lon2,
        ZOOM_LEVEL,
        default_prefs['url'],
        default_prefs['headers'],
        default_prefs['tile_size_pixels'],
        default_prefs['channels']
    )

    if OUTPUT_FORMAT == 'tif':
        import rasterio
        output_filename = f"{OUTPUT_DIR}/patch_{idx}_{lat}_{lon}.tif"
        # Calculate transform for GeoTIFF
        # The top-left corner of the image is (lon1, lat1)
        # The pixel size is the difference in degrees divided by the number of pixels
        transform = rasterio.transform.from_bounds(lon1, lat2, lon2, lat1, width, height)
        with rasterio.open(
            output_filename,
            'w',
            driver='GTiff',
            height=img.shape[0],
            width=img.shape[1],
            count=img.shape[2],
            dtype=img.dtype,
            crs='EPSG:4326',
            transform=transform,
        ) as dst:
            # Rasterio expects channels in (bands, height, width) format
            # OpenCV reads in (height, width, channels)
            for i in range(img.shape[2]):
                dst.write(img[:, :, i], i + 1)
        print(f"  Saved: {output_filename} (actual size: {img.shape[1]}x{img.shape[0]} pixels)")
    elif OUTPUT_FORMAT == 'png':
        output_filename = f"{OUTPUT_DIR}/patch_{idx:04d}_lat{lat:.6f}_lon{lon:.6f}.png"
        cv2.imwrite(output_filename, img)
        print(f"  Saved: {output_filename} (actual size: {img.shape[1]}x{img.shape[0]} pixels)")
    else:
        print(f"  Error: Unsupported output format '{OUTPUT_FORMAT}'. Please use 'tif' or 'png'.")


    # Optional: delay between requests to avoid rate limiting
    if REQUEST_DELAY > 0 and idx < len(df) - 1:
        import time
        time.sleep(REQUEST_DELAY)

    #break


print(f"\nCompleted! Downloaded {len(df)} satellite patches to '{OUTPUT_DIR}' directory.")

Processing 1/500: Center (23.944919, 85.543857)
  Bounding box: (23.946175, 85.542484) to (23.943664, 85.545230)
  Target image size: 512px x 512px at zoom level 18
  Ground coverage: ~279.4m x 279.4m (at 0.5458 m/pixel)
  Expected image size: 512x512 pixels
  Saved: satellite_patches/patch_0_23.9449194425092_85.54385681113175.tif (actual size: 512x512 pixels)
Processing 2/500: Center (24.288923, 88.455822)
  Bounding box: (24.290175, 88.454449) to (24.287671, 88.457196)
  Target image size: 512px x 512px at zoom level 18
  Ground coverage: ~278.7m x 278.7m (at 0.5443 m/pixel)
  Expected image size: 512x512 pixels
  Saved: satellite_patches/patch_1_24.28892298214672_88.45582227823141.tif (actual size: 512x512 pixels)
Processing 3/500: Center (23.506138, 86.745067)
  Bounding box: (23.507398, 86.743694) to (23.504879, 86.746440)
  Target image size: 512px x 512px at zoom level 18
  Ground coverage: ~280.4m x 280.4m (at 0.5476 m/pixel)
  Expected image size: 512x512 pixels
  Saved: satel

### Optional: download to local disk (in case you used Google Colab)

In [ ]:
import shutil
from google.colab import files

# Create a zip archive of the output directory
zip_filename = 'satellite_patches.zip'
shutil.make_archive(zip_filename.replace('.zip', ''), 'zip', OUTPUT_DIR)

# Download the zip file
files.download(zip_filename)